# Test Drift Detection
Simulate data drift to test monitoring capabilities

In [ ]:
import pandas as pd
import numpy as np
import json
import time
import boto3

In [ ]:
def generate_drifted_data(original_data, drift_factor=0.3):
    """Simulate data drift by shifting feature distributions"""
    drifted_data = original_data.copy()
    
    # Shift transaction amounts higher (economic inflation)
    drifted_data['transaction_amount'] *= (1 + drift_factor)
    
    # Increase late night transactions (behavior change)
    drifted_data['late_night_transaction'] += drift_factor
    
    # Change device scores (new devices)
    drifted_data['device_score'] += np.random.normal(0, drift_factor, len(drifted_data))
    
    return drifted_data

In [ ]:
# Load original test data
bucket = 'your-mlops-monitoring-bucket'  # Replace
test_data = pd.read_csv(f's3://{bucket}/data/fraud_dataset.csv').tail(1000)
feature_columns = [col for col in test_data.columns 
                  if col not in ['is_fraud', 'transaction_id', 'timestamp']]

print(f"Test data shape: {test_data.shape}")
print(f"Feature columns: {len(feature_columns)}")

In [ ]:
# Send normal predictions
endpoint_name = 'your-endpoint-name'  # Replace
runtime = boto3.client('sagemaker-runtime')

print("Sending normal data...")
for i in range(50):  # Reduced for demo
    sample = test_data[feature_columns].iloc[i:i+1]
    
    response = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType='text/csv',
        Body=sample.to_csv(index=False, header=False)
    )
    
    result = json.loads(response['Body'].read().decode())
    if i % 10 == 0:
        print(f"Prediction {i}: {result}")
    
    time.sleep(2)  # Rate limiting

In [ ]:
# Wait for monitoring to process
print("Waiting 5 minutes for monitoring...")
time.sleep(300)

In [ ]:
# Send drifted data
print("Sending drifted data...")
drifted_data = generate_drifted_data(test_data[feature_columns])

for i in range(50):  # Reduced for demo
    sample = drifted_data.iloc[i:i+1]
    
    response = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType='text/csv',
        Body=sample.to_csv(index=False, header=False)
    )
    
    result = json.loads(response['Body'].read().decode())
    if i % 10 == 0:
        print(f"Drifted prediction {i}: {result}")
    
    time.sleep(2)

print("Drift simulation complete! Check CloudWatch for alerts.")

In [ ]:
# Compare original vs drifted data statistics
print("Original data statistics:")
print(test_data[feature_columns].describe())

print("\nDrifted data statistics:")
print(drifted_data.describe())